# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one pseudonymized content item (page). 30,000 rows, one row per `content_id`, drawn from 32 pseudonymized `client_id`s.

**Time window:** every `*_90d` metric is a trailing 90-day window ending at export time — the same window for every row (no per-client windows here, unlike the warehouse release). Nested inside that are two 30-day sub-windows used for the trend proxy: `*_last_30d` (most recent 30 days) vs `*_prev_30d` (the 30 days before that, days 31–60 back). `content_age_days` is ≥ 90 for every row in this slice, so no row has a partially-filled window.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows, cols:", df.shape)
print("duplicate content_id (one row per page?):", df["content_id"].duplicated().sum())
print("distinct client_id:", df["client_id"].nunique())
print("content_age_days min/max:", df["content_age_days"].min(), "/", df["content_age_days"].max())
print("age_tier values present in this slice:", sorted(df["age_tier"].dropna().unique()))


rows, cols: (30000, 44)
duplicate content_id (one row per page?): 0
distinct client_id: 32
content_age_days min/max: 90 / 564
age_tier values present in this slice: ['181-365', '31-90', '365+', '91-180']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Buckets follow what the reference pipeline actually does with each column (`MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES` in `scripts/ml_utils.py`), not just what looks safe.

**Feature** (knowable before the moment we'd score a page, used by the pipeline):
- numeric: `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d`, `days_with_impressions`, `days_with_sessions`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- categorical: `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`

**Label / proxy** (never a feature): `trend_direction` and `trend_pct` → `is_declining_label = (trend_direction == "down")`. `trend_pct` is computed from `impressions_last_30d` vs `impressions_prev_30d`, and `trend_direction` is thresholded off `trend_pct`.

**Context** (grouping/joining/splitting only, never learned from): `content_id`, `client_id` — `client_id` specifically for client-holdout `GroupShuffleSplit`, so no client's pages leak across train/test.

**Excluded** (each with a one-line why):
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d` — raw heavy-tailed totals; the pipeline already uses their `log1p` versions, so keeping both double-counts the same signal.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — these are the literal inputs `trend_pct` is computed from (verified below); using them as features is near-total label leakage, not just correlation.
- `provider_used`, `model_used` — content-generation metadata the dictionary explicitly flags "not a model feature"; describes how the page was produced, not how it's performing.
- `age_tier_order`, `char_count_tier` — redundant re-encodings of `age_tier`/`content_age_days` and `char_count`/`word_count_tier`, which are already covered; unused by the reference pipeline.

In [2]:
raw_columns = set(df.columns)

feature_cols = {
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
}
# not raw columns -- scripts/01_prepare_features.py derives these from the excluded raw totals below
derived_features = {"log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d"}

label_cols = {"trend_direction", "trend_pct"}
context_cols = {"content_id", "client_id"}
excluded_cols = {
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "provider_used", "model_used", "age_tier_order", "char_count_tier",
}

classified = feature_cols | label_cols | context_cols | excluded_cols
buckets = [feature_cols, label_cols, context_cols, excluded_cols]
overlaps = sorted(set.union(*[a & b for i, a in enumerate(buckets) for b in buckets[i+1:]]))

print("raw columns in CSV:", len(raw_columns))
print("columns classified (feature+label+context+excluded):", len(classified))
print("raw columns left unclassified:", sorted(raw_columns - classified))
print("classified columns missing from raw CSV (should be empty -- confirms no typos):", sorted(classified - raw_columns))
print("columns claimed in more than one bucket (should be empty):", overlaps)
print("\nderived-only features (added by scripts/01_prepare_features.py, not present in the raw CSV):", sorted(derived_features))


raw columns in CSV: 44
columns classified (feature+label+context+excluded): 44
raw columns left unclassified: []
classified columns missing from raw CSV (should be empty -- confirms no typos): []
columns claimed in more than one bucket (should be empty): []

derived-only features (added by scripts/01_prepare_features.py, not present in the raw CSV): ['log_ai_sessions_90d', 'log_clicks_90d', 'log_impressions_90d', 'log_sessions_90d']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The contract claim that matters most here: `impressions_last_30d`/`impressions_prev_30d` aren't just correlated with the label — they ARE `trend_pct`'s formula. Recomputing it from the raw columns and comparing to the stored column proves it, rather than asserting it.

In [3]:
import numpy as np

# Grain
print("duplicate content_id (grain check):", df["content_id"].duplicated().sum())

# Counts
print("rows:", len(df), "| clients:", df["client_id"].nunique())
print("declining rate (trend_direction == 'down'):", round((df["trend_direction"] == "down").mean(), 3))

# Missing values -- check they follow content_type, not random
print("\nsearch_volume missing rate by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: round(s.isna().mean(), 3)))
print("\nword_count missing rate by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: round(s.isna().mean(), 3)))

# Sentinel values that look like real numbers but aren't
print("\navg_position == 0 ('no data', not rank zero):", (df["avg_position"] == 0).sum())
print("scroll_rate > 100 (multiple scroll events per pageview, not a bug):", (df["scroll_rate"] > 100).sum())
print("ai_traffic_pct > 100 (independent measurement systems, not a bug):", (df["ai_traffic_pct"] > 100).sum())

# Windows
print("\ntrend_pct missing rows:", df["trend_pct"].isna().sum(),
      "| impressions_prev_30d == 0 rows:", df["impressions_prev_30d"].eq(0).sum())

# Leakage proof: recompute trend_pct from its declared inputs and compare to the stored column
recomputed = ((df["impressions_last_30d"] - df["impressions_prev_30d"])
              / df["impressions_prev_30d"].replace(0, np.nan) * 100)
comparable = df["trend_pct"].notna() & recomputed.notna()
match = np.isclose(recomputed[comparable], df.loc[comparable, "trend_pct"], atol=0.05)
print("\ntrend_pct recomputed-from-raw-columns match rate:", round(match.mean(), 3),
      f"(n={comparable.sum()}) -- confirms last_30d/prev_30d ARE the label's inputs, not just correlated")


duplicate content_id (grain check): 0
rows: 30000 | clients: 32
declining rate (trend_direction == 'down'): 0.542

search_volume missing rate by content_type:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64

word_count missing rate by content_type:
content_type
comparison article    0.000
feedly article        0.000
keyword article       0.283
Name: word_count, dtype: float64

avg_position == 0 ('no data', not rank zero): 1205
scroll_rate > 100 (multiple scroll events per pageview, not a bug): 119
ai_traffic_pct > 100 (independent measurement systems, not a bug): 23

trend_pct missing rows: 3388 | impressions_prev_30d == 0 rows: 3388

trend_pct recomputed-from-raw-columns match rate: 1.0 (n=26612) -- confirms last_30d/prev_30d ARE the label's inputs, not just correlated


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No real future outcome.** `is_declining_label` is a proxy from a 30-vs-30-day comparison inside one 90-day snapshot, not an observed outcome that happened after a decision. Nothing here shows that refreshing a flagged page actually recovers it — that needs a later window this file doesn't have.
- **One frozen slice, not a panel.** There's no date/timestamp column (checked below) — every row is a single export-time snapshot, so *when* a decline started or how fast it's moving isn't recoverable from this CSV alone (the warehouse's `fact_content_daily_performance` has that history).
- **A sample, not the population.** 30,000 rows across 32 clients here vs. the documented full warehouse release of 519,606 content items across 104 clients (`docs/data-dictionary.md`) — rates measured on this slice (e.g. the 54.2% decline rate) describe this teaching slice, not every client.
- **Missingness tracks content_type, not chance** (verified above) — any comparison leaning on `search_volume`/`word_count` inherits a content-type mix bias unless that's controlled for.
- **No causal claims.** No experiment, no randomization, no control group — anything this data supports is observed / directional / decision-support, never "refreshing this page will improve X."

In [4]:
import re

print("columns in raw CSV:", len(df.columns))

# single snapshot, not a panel: no per-day date/timestamp column in the raw export
# (word-boundary match -- a naive substring check wrongly flags days_since_last_update, which
# contains "update", not a date column)
date_like = [c for c in df.columns if re.search(r"\bdate\b|\btimestamp\b", c.lower())]
print("date-like columns:", date_like)

# this slice vs. the documented warehouse population (docs/data-dictionary.md, not queryable from this CSV)
print(f"this slice: {df['client_id'].nunique()} clients, {len(df)} rows")
print("documented warehouse population: 104 clients, 519,606 content items (dim_content)")


columns in raw CSV: 44
date-like columns: []
this slice: 32 clients, 30000 rows
documented warehouse population: 104 clients, 519,606 content items (dim_content)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.